In [ ]:
pip list

In [3]:
# ==================================================
# 1. IMPORTS
# ==================================================
import pandas as pd
import numpy as np
import os
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

import scorecardpy as sc

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==================================================
# 2. PATH
# ==================================================
MODEL_PATH = r"C:/Users/Fernanda Pavan/OneDrive/Desktop/Projeto_Risco_Credito/models"
os.makedirs(MODEL_PATH, exist_ok=True)

# ==================================================
# 3. LOAD DATA
# ==================================================
df = pd.read_csv(
    r"C:/Users/Fernanda Pavan/OneDrive/Desktop/Projeto_Risco_Credito/data/german_credit_data.csv"
)

# ==================================================
# 4. FEATURE ENGINEERING
# ==================================================
df1 = df.copy()

df1['Saving accounts'] = df1['Saving accounts'].fillna('little')
df1['Checking account'] = df1['Checking account'].fillna('little')
df1['Saving accounts'] = df1['Saving accounts'].replace('quite rich', 'rich')

df1.rename(columns={
    'Age': 'Idade',
    'Sex': 'Genero',
    'Job': 'Trabalho',
    'Housing': 'Habitacao',
    'Saving accounts': 'Conta_poupanca',
    'Checking account': 'Conta_corrente',
    'Credit amount': 'Valor_credito',
    'Duration': 'Duracao',
    'Purpose': 'Finalidade',
    'Risk': 'Risco'
}, inplace=True)

# ==================================================
# 5. BINNING
# ==================================================
df1["Faixa_Etaria"] = pd.cut(
    df1["Idade"],
    [0,28,38,48,58,68,78]
).astype(str)

df1["Faixa_Duracao"] = pd.cut(
    df1["Duracao"],
    [0,15,27,39,51,63,75]
).astype(str)

df1["Faixa_Credito"] = pd.cut(
    df1["Valor_credito"],
    range(250,20000,1000)
).astype(str)

# ==================================================
# 6. TARGET
# ==================================================
df1["Risco"] = df1["Risco"].map({
    "good":1,
    "bad":0
})

# ==================================================
# 7. WOE
# ==================================================
df_woe = df1.copy()

df_woe = df_woe.drop(
    columns=[
        "Idade",
        "Duracao",
        "Valor_credito",
        "Unnamed: 0"
    ],
    errors="ignore"
)

for col in [
    "Faixa_Etaria",
    "Faixa_Duracao",
    "Faixa_Credito"
]:
    df_woe[col] = df_woe[col].astype(str)

bins = sc.woebin(
    df_woe,
    y="Risco"
)

joblib.dump(
    bins,
    os.path.join(
        MODEL_PATH,
        "woe_bins.pkl"
    )
)

df_woe = sc.woebin_ply(
    df_woe,
    bins
)

feature_names = df_woe.drop(
    "Risco",
    axis=1
).columns.tolist()

# ==================================================
# 8. SPLIT
# ==================================================
X = df_woe.drop(
    "Risco",
    axis=1
)

y = df_woe["Risco"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# ==================================================
# 9. MODELOS
# ==================================================
models = {

    "Logistic":

        LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ),

    "RandomForest":

        RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            class_weight="balanced",
            random_state=42
        ),

    "XGBoost":

        XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            eval_metric="logloss",
            random_state=42
        )
}

# ==================================================
# 10. TREINAMENTO
# ==================================================
results=[]
trained_models={}

for name, model in models.items():

    model.fit(
        X_train,
        y_train
    )

    prob=model.predict_proba(
        X_test
    )[:,1]

    pred=(prob>=0.5).astype(int)

    auc=roc_auc_score(
        y_test,
        prob
    )

    gini=2*auc-1

    fpr,tpr,_=roc_curve(
        y_test,
        prob
    )

    ks=np.max(
        tpr-fpr
    )

    acc=accuracy_score(
        y_test,
        pred
    )

    prec=precision_score(
        y_test,
        pred
    )

    rec=recall_score(
        y_test,
        pred
    )

    f1=f1_score(
        y_test,
        pred
    )

    results.append({

        "Modelo":name,
        "Accuracy":acc,
        "Precision":prec,
        "Recall":rec,
        "F1":f1,
        "AUC":auc,
        "Gini":gini,
        "KS":ks

    })

    trained_models[name]=model

# ==================================================
# 11. MELHOR MODELO
# ==================================================
results_df=pd.DataFrame(
    results
).sort_values(
    by="KS",
    ascending=False
)

best_model_name=results_df.iloc[0]["Modelo"]

best_model=trained_models[
    best_model_name
]

# ==================================================
# 12. PREVISÃO FINAL
# ==================================================
y_prob=best_model.predict_proba(
    X_test
)[:,1]

y_pred=(y_prob>=0.5).astype(int)

accuracy=accuracy_score(
    y_test,
    y_pred
)

precision=precision_score(
    y_test,
    y_pred
)

recall=recall_score(
    y_test,
    y_pred
)

f1=f1_score(
    y_test,
    y_pred
)

auc=roc_auc_score(
    y_test,
    y_prob
)

gini=2*auc-1

fpr,tpr,_=roc_curve(
    y_test,
    y_prob
)

ks=np.max(
    tpr-fpr
)

cm=confusion_matrix(
    y_test,
    y_pred
)

# ==================================================
# 13. FEATURE IMPORTANCE
# ==================================================
if hasattr(
    best_model,
    "feature_importances_"
):

    features=pd.DataFrame({

        "feature":X.columns,
        "importance":
            best_model.feature_importances_

    }).sort_values(
        by="importance",
        ascending=False
    )

else:

    features=pd.DataFrame({

        "feature":X.columns,
        "importance":
            best_model.coef_[0]

    }).sort_values(
        by="importance",
        ascending=False
    )

# ==================================================
# 14. RESULTADOS
# ==================================================
resultados=X_test.copy()

resultados["y_true"]=y_test.values

resultados["y_prob"]=y_prob

resultados["y_pred"]=y_pred

# ==================================================
# 15. SALVAR ARQUIVOS
# ==================================================
joblib.dump(
    best_model,
    os.path.join(
        MODEL_PATH,
        "modelo.pkl"
    )
)

joblib.dump(
    feature_names,
    os.path.join(
        MODEL_PATH,
        "feature_names.pkl"
    )
)

results_df.to_csv(
    os.path.join(
        MODEL_PATH,
        "comparacao_modelos.csv"
    ),
    index=False
)

resultados.to_csv(
    os.path.join(
        MODEL_PATH,
        "resultados_modelo.csv"
    ),
    index=False
)

features.to_csv(
    os.path.join(
        MODEL_PATH,
        "features_importancia.csv"
    ),
    index=False
)

cm_df=pd.DataFrame(
    cm,
    columns=["Pred_0","Pred_1"],
    index=["Real_0","Real_1"]
)

cm_df.to_csv(
    os.path.join(
        MODEL_PATH,
        "matriz_confusao.csv"
    )
)

# ==================================================
# 16. MÉTRICAS JSON
# ==================================================
metricas={

    "best_model":best_model_name,

    "accuracy":float(accuracy),

    "precision":float(precision),

    "recall":float(recall),

    "f1_score":float(f1),

    "auc":float(auc),

    "gini":float(gini),

    "ks":float(ks),

    "confusion_matrix":{

        "TN":int(cm[0][0]),
        "FP":int(cm[0][1]),
        "FN":int(cm[1][0]),
        "TP":int(cm[1][1])

    }

}

with open(
    os.path.join(
        MODEL_PATH,
        "metricas.json"
    ),
    "w"
) as f:

    json.dump(
        metricas,
        f,
        indent=4
    )

# ==================================================
# 17. PDO SCORECARD (NOVO)
# ==================================================
score_params = {

    "pdo":20,

    "base_score":600,

    "base_odds":50

}

with open(
    os.path.join(
        MODEL_PATH,
        "score_params.json"
    ),
    "w"
) as f:

    json.dump(
        score_params,
        f,
        indent=4
    )

# ==================================================
# FINAL
# ==================================================
print("\nPIPELINE FINALIZADO")
print("Modelo vencedor:",best_model_name)
print("Métricas salvas ✔")
print("Parâmetros PDO salvos ✔")

[INFO] creating woe binning ...
[INFO] converting into woe values ...

PIPELINE FINALIZADO
Modelo vencedor: Logistic
Métricas salvas ✔
Parâmetros PDO salvos ✔
